# 03 — GRU

Notebook ini berdiri sendiri: jalankan `make preprocess` sekali dulu
(datanya diambil dari BigQuery), lalu Run All di sini untuk train + evaluasi
+ upload model GRU saja.

Input-nya tabel `preprocessed_data_hourly` + `preprocessing_metadata_hourly`
hasil `make preprocess`: semua stasiun + kalender (sin/cos) untuk semua model,
ditambah akumulasi hujan DAS milik stasiun target. `USE_RAINFALL = False`
menjalankan ablation tanpa hujan; hasilnya disimpan sebagai `<model>_norain`.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

# False = ablation tanpa fitur hujan; hasil disimpan sebagai <model>_norain.
USE_RAINFALL = True

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_hourly"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_hourly"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
station_cols  = metadata["station_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}

# Fitur: semua stasiun + kalender (sin/cos) untuk semua model, ditambah
# akumulasi hujan DAS milik stasiun target (lihat `make preprocess`).
if "base_feature_cols" not in metadata:
    raise RuntimeError("Metadata versi lama (tanpa fitur hujan/kalender sin-cos); jalankan `make preprocess` dulu.")
BASE_FEATURE_COLS    = metadata["base_feature_cols"]
RAIN_COLS_BY_STATION = metadata["rain_cols_by_station"] if USE_RAINFALL else {}
FEATURES_BY_STATION  = {s: BASE_FEATURE_COLS + RAIN_COLS_BY_STATION.get(s, []) for s in station_cols}
ALL_FEATURE_COLS     = BASE_FEATURE_COLS + [c for cols in RAIN_COLS_BY_STATION.values() for c in cols]


def features_for(target):
    """Kolom input model untuk stasiun target (urutan sama dengan serving)."""
    return FEATURES_BY_STATION[target]


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[ALL_FEATURE_COLS]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
n_rain = {s: len(c) for s, c in RAIN_COLS_BY_STATION.items()}
print(f"Features : {len(BASE_FEATURE_COLS)} dasar + fitur hujan per stasiun: {n_rain} (USE_RAINFALL={USE_RAINFALL})")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `gru_val_metrics_hourly` — metrik validasi per (station, horizon, metric).
- `gru_weights_hourly` — bobot terlatih per (stasiun, horizon), disimpan sebagai `BYTES`.
- `gru_predictions_hourly` — prediksi test-set per (station, horizon, datetime).
- `gru_metrics_hourly` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "gru" if USE_RAINFALL else "gru_norain"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_hourly"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_hourly"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_hourly"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_hourly"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK   = 72      # jam
HORIZONS   = [1, 3, 12]  # jam ke depan
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 7       # early stopping

print(f"Device     : {DEVICE}")
print(f"Lookback   : {LOOKBACK} jam")
print(f"Models     : {len(station_cols)} stasiun \u00d7 {len(HORIZONS)} horizon = {len(station_cols)*len(HORIZONS)}")

In [ ]:
class TMADataset(Dataset):
    def __init__(self, df, target_station, feature_cols, lookback, horizon):
        self.X, self.y = [], []
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)

        for i in range(lookback, len(data) - horizon + 1):
            self.X.append(data[i - lookback:i])          # (lookback, n_features)
            self.y.append(target[i + horizon - 1])        # scalar t+h

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)   # ambil timestep terakhir

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            total_loss += criterion(pred, y).item() * len(y)
    return total_loss / len(loader.dataset)


def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).cpu().numpy())
            trues.append(y.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
gru_models = {}   # {station: {horizon: model}}
gru_results = {}   # {station: {horizon: metrics}}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    gru_models[target] = {}
    gru_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, features_for(target), LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, features_for(target), LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = GRUForecaster(n_features=len(features_for(target))).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        gru_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        gru_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]


def serialize_torch_state(state_dict):
    """Serialize state_dict PyTorch ke bytes (in-memory, tanpa file sementara)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getvalue()


saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, horizons in gru_models.items():
    for h, model in horizons.items():
        blob = serialize_torch_state(model.state_dict())
        weight_rows.append({
            "model": MODEL_NAME, "station": station, "horizon": int(h),
            "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
        })
    for hkey, metrics in gru_results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

In [ ]:
def _load_weights_from_bq(station, h):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station AND horizon = @horizon
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
        bigquery.ScalarQueryParameter("horizon", "INT64", h),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot utk ({station}, h+{h}) tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_gru(target, h):
    test_ds = TMADataset(df_test, target, features_for(target), LOOKBACK, h)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    if target in gru_models and h in gru_models[target]:
        model = gru_models[target][h]
    else:
        # rebuild + load bobot dari BigQuery (mis. notebook baru dibuka ulang)
        model = GRUForecaster(n_features=len(features_for(target))).to(DEVICE)
        blob = _load_weights_from_bq(target, h)
        model.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))

    idx = df_test.index[LOOKBACK + h - 1 : len(df_test)]
    y_pred, y_true = get_predictions(model, test_loader)
    return idx[:len(y_pred)], y_true, y_pred


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_gru(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": MODEL_NAME,
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": MODEL_NAME, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")